In [1]:
import dask
import subprocess
from dask_jobqueue import PBSCluster
from dask.distributed import Client, get_worker

!rm -f dask-fsscan.* foo_out.Worker-*

In [2]:
# Create a PBS cluster object
cluster = PBSCluster(
    job_name = 'dask-fsscan',
    cores = 1,
    memory = '1GiB',
    processes = 1,
    local_directory = '/local_scratch/pbs.${PBS_JOBID}/dask/spill',
    resource_spec = 'select=1:ncpus=1:mem=1GB',
    queue = 'casper',
    walltime = '2:00:00',
    interface = 'ext',
    job_extra_directives = [ '-j oe' ]
)

/glade/u/apps/opt/conda/envs/npl-2025a/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42735 instead
  warnings.warn(


In [3]:
print(cluster.job_script())

#!/usr/bin/env bash

#PBS -N dask-fsscan
#PBS -q casper
#PBS -A SCSG0001
#PBS -l select=1:ncpus=1:mem=1GB
#PBS -l walltime=2:00:00
#PBS -j oe

/glade/u/apps/opt/conda/envs/npl-2025a/bin/python -m distributed.cli.dask_worker tcp://128.117.208.119:44731 --name dummy-name --nthreads 1 --memory-limit 1.00GiB --nanny --death-timeout 60 --local-directory /local_scratch/pbs.${PBS_JOBID}/dask/spill --interface ext



In [4]:
# Create the client to load the Dashboard
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/benkirk/proxy/42735/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/benkirk/proxy/42735/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.119:44731,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/benkirk/proxy/42735/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [5]:
n_workers = 64
cluster.scale(n_workers)
client.wait_for_workers(2)

In [6]:
def scan_subdir(dirname, recursive):
    id = get_worker().id    

    command = ['find', '{}'.format(dirname) ]
    if not recursive:
        command.extend([ '-maxdepth', '1', '-mindepth', '1' ])
        
    command.extend([ '-type', 'f',
                     '-size', '+2G',
                     '-atime', '+{:d}'.format(3*365),
                     '-printf',  '%20s %AF %12u %12g --- %p\\n' ])
    
    filename='foo_out.{}'.format(id)

    with open(filename, 'a') as ofile:
        process = subprocess.Popen(command, 
                                   stdout=subprocess.PIPE, 
                                   stderr=subprocess.DEVNULL, 
                                   text=True,
                                   errors='ignore',
                                   bufsize=1)

        for line in iter(process.stdout.readline, ''):                
            ofile.write(line)
            
        process.stdout.close()
        return_code = process.wait()
            
    return

In [7]:
def run_scan():
    
    recursive=True
    for depth in range(4,0,-1):
        command = 'find /glade/campaign -maxdepth {0} -mindepth {0} -type d'.format(depth)
        print(command)
        
        process = subprocess.Popen(command.split(), 
                                   stdout=subprocess.PIPE, 
                                   stderr=subprocess.DEVNULL, 
                                   text=True,
                                   errors='ignore',
                                   bufsize=1)
        
        for dir in iter(process.stdout.readline, ''):
            dir = dir.strip()
            client.submit(scan_subdir, dir, recursive)
            
        process.stdout.close()
        return_code = process.wait()

        recursive = False
        
run_scan()
print('done')

find /glade/campaign -maxdepth 4 -mindepth 4 -type d
find /glade/campaign -maxdepth 3 -mindepth 3 -type d
find /glade/campaign -maxdepth 2 -mindepth 2 -type d
find /glade/campaign -maxdepth 1 -mindepth 1 -type d
done


In [8]:
# Shut down our client (thus terminating workers)
#client.shutdown()

In [20]:
import time

current_timestamp = time.time()
print(current_timestamp)

import os
for di in os.scandir('.'):
    pathname = di.path
    statinfo = di.stat(follow_symlinks=False)
    print(pathname, statinfo)
    print(current_timestamp - statinfo.st_atime)
    break

1752013223.1766763
./dask-fsscan.o5557520 os.stat_result(st_mode=33188, st_ino=5053848051, st_dev=66, st_nlink=1, st_uid=38057, st_gid=1000, st_size=2244, st_atime=1752005844, st_mtime=1752013074, st_ctime=1752013074)
7379.013984203339
